# PrimeTrade — Bitcoin Sentiment × Trader Performance Analysis

**Hyperliquid Historical Data × Fear & Greed Index**

Exploring the relationship between market sentiment and trader performance to uncover actionable trading strategy insights.

## Contents
1. [Data Loading](#1)
2. [Sentiment Distribution](#2)
3. [PnL by Sentiment Regime](#3)
4. [Leverage Behavior](#4)
5. [Long/Short Positioning](#5)
6. [Top Trader Profiling](#6)
7. [Symbol × Sentiment Heatmap](#7)
8. [Lag Correlation](#8)
9. [Contrarian Strategy Backtest](#9)
10. [Sentiment Timeline](#10)
11. [Transition Matrix](#11)
12. [ML Model](#12)
13. [Key Insights](#13)

In [ ]:
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.rcParams["figure.dpi"] = 120

from config.settings import FIGURES_DIR, SENTIMENT_COLORS, SENTIMENT_ORDER
from src.analysis.correlation import (
    compute_lag_correlations,
    contrarian_backtest,
    rolling_sentiment_momentum,
    sentiment_transition_matrix,
)
from src.analysis.trader_metrics import (
    daily_pnl_timeseries,
    leverage_by_sentiment,
    long_short_ratio_by_sentiment,
    pnl_by_sentiment,
    pnl_by_symbol_sentiment,
    statistical_tests,
    top_traders,
    trader_sentiment_preference,
)
from src.ingestion.loader import load_and_merge, load_fear_greed, validate_merged
from src.models.sentiment_predictor import SentimentTradePredictor, plot_model_results
from src.visualization.charts import *

## 1. Data Loading <a id='1'></a>

In [ ]:
df = load_and_merge(save=True)
fg = load_fear_greed()
stats = validate_merged(df)

print(f"Total records    : {stats['total_rows']:,}")
print(f"CLOSE events     : {stats['close_events']:,}")
print(f"Unique accounts  : {stats['unique_accounts']}")
print(f"Unique symbols   : {stats['unique_symbols']}")
print(f"Date range       : {stats['date_range'][0]} → {stats['date_range'][1]}")
print()
print("Sentiment distribution (trade days):")
for k, v in stats["sentiment_coverage"].items():
    bar = "█" * (v // 50)
    print(f"  {k:<15}: {v:4d}  {bar}")

In [ ]:
display(df.head(8))
display(df.dtypes.to_frame("dtype"))

## 2. Sentiment Distribution <a id='2'></a>

In [ ]:
fig = plot_sentiment_distribution(fg)
plt.show()

fg_plot = fg.copy()
if "value" not in fg_plot.columns:
    enc = {s: i * 25 for i, s in enumerate(SENTIMENT_ORDER)}
    fg_plot["value"] = fg_plot["classification"].map(enc)

fg_plot = fg_plot.sort_values("date")
fg_plot["rolling_30d"] = fg_plot["value"].rolling(30, min_periods=1).mean()

plt.figure(figsize=(14, 4), facecolor="#0d0d1a")
ax = plt.gca()
ax.set_facecolor("#0d0d1a")
ax.plot(fg_plot["date"], fg_plot["value"], color="#444", lw=0.8, alpha=0.5)
ax.plot(fg_plot["date"], fg_plot["rolling_30d"], color="#00d4ff", lw=2, label="30-day rolling avg")
ax.axhline(50, color="#666", lw=0.8, linestyle="--", label="Neutral line")
ax.fill_between(fg_plot["date"], fg_plot["value"], 50,
    where=fg_plot["value"] < 50, alpha=0.2, color="#d62728", label="Fear zone")
ax.fill_between(fg_plot["date"], fg_plot["value"], 50,
    where=fg_plot["value"] > 50, alpha=0.2, color="#2ca02c", label="Greed zone")
ax.set_title("Fear & Greed Index Over Time", color="#00d4ff", fontsize=12)
ax.legend(fontsize=8)
ax.set_ylim(0, 100)
ax.tick_params(colors="#a0a0c0")
plt.tight_layout()
plt.show()

## 3. PnL by Sentiment Regime <a id='3'></a>

In [ ]:
pnl_df = pnl_by_sentiment(df)
display(pnl_df.round(3))

stat_tests = statistical_tests(df)
print(f"Kruskal-Wallis: H={stat_tests['kruskal_wallis']['stat']:.4f}, p={stat_tests['kruskal_wallis']['p']:.4f}")
sig = "Significant" if stat_tests["kruskal_wallis"]["p"] < 0.05 else "Not significant"
print(f"  → {sig} difference across regimes")
print()
display(stat_tests["pairwise"])

In [ ]:
fig = plot_pnl_by_sentiment(pnl_df)
plt.show()

In [ ]:
fig = plot_pnl_violin(df)
plt.show()

## 4. Leverage Behavior <a id='4'></a>

In [ ]:
lev_df = leverage_by_sentiment(df)
display(lev_df)
fig = plot_leverage_by_sentiment(lev_df)
plt.show()

## 5. Long/Short Positioning <a id='5'></a>

In [ ]:
ls_df = long_short_ratio_by_sentiment(df)
display(ls_df)
fig = plot_long_short_ratio(ls_df)
plt.show()

## 6. Top Trader Profiling <a id='6'></a>

In [ ]:
top_df = top_traders(df, top_n=10)
display(top_df[["account", "total_pnl", "win_rate", "trade_count", "sharpe"]].round(3))
fig = plot_top_traders(top_df)
plt.show()

In [ ]:
pref = trader_sentiment_preference(df)
top_accts = top_df["account"].values[:5]
pref_top = pref[pref["account"].isin(top_accts)]
display(pref_top.pivot_table(index="account", columns="classification", values="avg_pnl").round(2))

## 7. Symbol × Sentiment Heatmap <a id='7'></a>

In [ ]:
symbol_pivot = pnl_by_symbol_sentiment(df)
display(symbol_pivot.round(2))
fig = plot_symbol_sentiment_heatmap(symbol_pivot)
plt.show()

## 8. Lag Correlation: Sentiment → PnL <a id='8'></a>

In [ ]:
lag_df = compute_lag_correlations(df)
display(lag_df)
fig = plot_lag_correlation(lag_df)
plt.show()

momentum = rolling_sentiment_momentum(df, window=7)
display(momentum.tail(10).round(3))

## 9. Contrarian Strategy Backtest <a id='9'></a>

In [ ]:
backtest = contrarian_backtest(df)
for k, v in backtest["summary"].items():
    print(f"  {k:<30}: {v}")
fig = plot_contrarian_strategy(backtest)
plt.show()

## 10. Sentiment × PnL Timeline <a id='10'></a>

In [ ]:
fig = plot_sentiment_timeline(fg, df)
plt.show()

## 11. Sentiment Transition Matrix <a id='11'></a>

In [ ]:
transition = sentiment_transition_matrix(fg)
display(transition.round(3))
fig = plot_transition_matrix(transition)
plt.show()

## 12. ML Model: Trade Outcome Predictor <a id='12'></a>

In [ ]:
predictor = SentimentTradePredictor()
predictor.fit(df)
results = predictor.evaluate()

print(f"ROC-AUC (test)  : {results['roc_auc']:.4f}")
print(f"ROC-AUC (CV)    : {results['cv_auc_mean']:.4f} ± {results['cv_auc_std']:.4f}")
print()
cr = pd.DataFrame(results["classification_report"]).T
display(cr.round(3))

plot_model_results(results, FIGURES_DIR)
display(predictor.feature_importance_df().round(4))

## 13. Key Insights & Recommendations <a id='13'></a>

### Greed regimes are most profitable on average
Mean PnL peaks during Greed, but compresses at Extreme Greed — diminishing momentum returns. Scale into positions as greed builds; reduce exposure at extremes.

### Leverage peaks at Greed with unfavourable risk/reward
Elevated leverage during Extreme Greed combined with compressed returns creates poor risk-adjusted outcomes. Cap leverage at 10× during Extreme Greed.

### Long bias strengthens with greed (rational)
Long/short ratio increases monotonically with sentiment. In Extreme Fear, short crowding creates contrarian long opportunities.

### Lag correlation is weak but present at 0–2 days
Pearson r ~0.035 at lag 0, decaying to near zero by 7 days. Sentiment has a short-lived, mild predictive signal useful for intraday/next-day positioning.

### Pure contrarianism is insufficient
The simple buy-fear/sell-greed strategy underperforms without timing filters (momentum confirmation, volume signals).

### Symbol variation is significant
ETH and SOL show the most consistent PnL differences across sentiment regimes. Tilt portfolio toward these for sentiment-driven strategies.

### ML model confirms sentiment as a secondary signal
ROC-AUC ~0.87. Trade size and hour-of-day are stronger predictors than sentiment alone — execution timing and position sizing dominate.

### Recommended Strategy Framework

| Regime | Action | Leverage Cap | Direction Bias |
|--------|--------|--------------|----------------|
| Extreme Fear | Accumulate Longs | 5× | Long |
| Fear | Cautious Longs | 3× | Neutral/Long |
| Neutral | Hold / Reduce | 2× | Flat |
| Greed | Ride trend | 5× | Long |
| Extreme Greed | Take profit / Hedge | 2× | Reduce/Short |